In [1]:
from transformers import pipeline,AutoModelForMaskedLM,Trainer,TrainingArguments,AutoTokenizer
from datasets import load_dataset
from huggingface_hub import notebook_login
import numpy as np
import pandas as pd
import torch
from collections import defaultdict

/opt/anaconda3/envs/tf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_checkpoint = "distilbert-base-uncased"
model=AutoModelForMaskedLM.from_pretrained(model_checkpoint)

Loading weights: 100%|█████████████████████| 105/105 [00:00<00:00, 6960.46it/s]


In [3]:
model.num_parameters()/1_000_000

66.98553

In [4]:
tokenizer=AutoTokenizer.from_pretrained(model_checkpoint)

In [5]:
text = "This is a great [MASK]."
inputs=tokenizer(text,return_tensors="pt")

In [6]:
inputs

{'input_ids': tensor([[ 101, 2023, 2003, 1037, 2307,  103, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

In [7]:
output=model(**inputs)

In [8]:
token_logits=output.logits

In [9]:
mask_token_idx=torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

In [10]:
mask_token_logits=token_logits[0,mask_token_idx,:]
top5_tokens=torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()
top5_tokens

[3066, 3112, 6172, 2801, 8658]

In [11]:
for token in top5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

'>>> This is a great deal.'
'>>> This is a great success.'
'>>> This is a great adventure.'
'>>> This is a great idea.'
'>>> This is a great feat.'


In [12]:
dataset=load_dataset("stanfordnlp/imdb")

In [13]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [14]:
df=dataset['train'].to_pandas()

In [15]:
df.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [16]:
def tokenize_dataset_fn(dataset):
    tokenized_dataset=tokenizer(dataset["text"])
    if tokenizer.is_fast:
        tokenized_dataset["word_ids"]=[tokenized_dataset.word_ids(i) for i in range(len(tokenized_dataset["input_ids"]))]
    return tokenized_dataset

In [17]:
tokenized_dataset=dataset.map(tokenize_dataset_fn,batched=True,remove_columns=dataset["train"].column_names)

In [18]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 50000
    })
})

In [19]:
tokenizer.model_max_length

512

In [20]:
chunk_size=128

In [21]:
def group_texts_and_chunk(dataset):
    concatenated_dataset={k:sum(dataset[k],[]) for k in dataset.keys()}
    total_length=len(concatenated_dataset[list(dataset.keys())[0]])
    total_length = (total_length // chunk_size) * chunk_size
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_dataset.items()
    }
    result["labels"]=result["input_ids"].copy()
    return result
    

In [22]:
dataset=tokenized_dataset.map(group_texts_and_chunk,batched=True)
dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 61291
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 59904
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 122957
    })
})

In [23]:
from transformers import default_data_collator

In [24]:
wwm_prob=0.15

In [25]:
def whole_word_masking(features):
    for feature in features:
        word_ids=feature.pop("word_ids",None)
        if word_ids is None:
            continue
        mapping=collections.defaultdict(list)
        curr_word_idx=-1
        curr_word=None

        for idx,word_id in enumerate(word_ids):
            if word_id is not None:
                if word_id!=curr_word:
                    curr_word=word_id
                    curr_word_idx+=1
                mapping[curr_word_idx].append(idx)

        mask=np.binomial(1,wwm_prob,(len(mapping),))
        input_ids=feature["input_ids"]
        labels=feature["labels"]
        new_labels=[-100]*len(labels)

        for word_id in np.where(mask)[0]:
            word_id=word_id.item()
            for idx in mapping[word_id]:
                new_labels[idx]=labels[idx]
                input_ids[idx]=tokenizer.mask_token_id

        feature["labels"]=new_labels
    return default_data_collator(features)
    

In [26]:
def compute_metrics(eval_preds):
    logits,labels=eval_preds
    logits=torch.tensor(logits)
    labels=torch.tensor(labels)
    criterion=nn.CrossEntropyLoss()
    loss=criterion(logits.view(-1,logits.size(-1),labels.view(-1)))
    perplexity=np.exp(loss.item())

    return {"perplexity": perplexity}

In [27]:
args=TrainingArguments("distilbert-finetuned-imdb",eval_strategy="epoch",save_strategy="epoch",num_train_epochs=1,logging_strategy="steps",logging_steps=50,
                       learning_rate=2e-5,weight_decay=0.01,push_to_hub=True,fp16=True,report_to="wandb")

In [28]:
trainer=Trainer(model=model,args=args,train_dataset=dataset['train'],eval_dataset=dataset['test'],data_collator=whole_word_masking,processing_class=tokenizer,compute_metrics=compute_metrics)

In [29]:
import wandb
wandb.login()

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/adnaniqbalkantroo/.netrc.
wandb: Currently logged in as: romancobblepot (romancobblepot-iit-kharagpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
trainer.train()

/opt/anaconda3/envs/tf/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [1]:
trainer.push_to_hub()

NameError: name 'trainer' is not defined